# Sensitivity and uncertainty

Built with [Vicena](https://vicena.ai). Local sensitivities and synthetic Monte Carlo intervals are screening tools.

In [1]:
import sys
from pathlib import Path
root=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd(); sys.path.insert(0,str(root/'src'))
import numpy as np
from chemical_reactor_twin import ReactionNetwork, simulate_batch
from chemical_reactor_twin.analysis import local_sensitivity, uncertainty_propagation

In [2]:
def conversion(p):
 net=ReactionNetwork(['A','B'],[[-1,1]],[p['A']],[p['Ea']],[[1,0]],[-p['dH']])
 r=simulate_batch(net,[1000,0],330,30,adiabatic=True)
 return [1-r.concentration[-1,0]/1000,r.temperature.max()]
s=local_sensitivity(conversion,{'A':10.0,'Ea':10000.0,'dH':15000.0})
print('base conversion, peak T',s['base']); print('derivatives',s['derivatives'])
def samples(rng):
 for _ in range(100): yield {'A':rng.lognormal(np.log(10),0.08),'Ea':rng.normal(10000,300),'dH':rng.normal(15000,500)}
u=uncertainty_propagation(conversion,samples,lambda x:x)
print('uncertainty mean',u['mean']); print('q05',u['q05']); print('q95',u['q95']); print('n',u['n_samples'])

base conversion, peak T [  0.99970054 333.58744211]
derivatives {'A': array([0.00024415, 0.00087614]), 'Ea': array([-8.81523643e-07, -3.16336236e-06]), 'dH': array([5.53205298e-09, 2.39182659e-04])}


uncertainty mean [  0.99953376 333.57794396]
q05 [  0.99866277 333.3830474 ]
q95 [  0.99995161 333.7388194 ]
n 100
